# Speculative decoding: Kaggle runner

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, and a Kaggle Secret holding a Hugging Face read token (named `kaggle` here; change `SECRET_NAME` in cell 4 if yours differs) attached to this notebook.

Run top to bottom, or use **Save Version → Save & Run All** for a recorded run.

In [ ]:
# 1. Hardware check: fail fast if Kaggle gave us the wrong machine.
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU. Set Accelerator to 'GPU T4 x2'."
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(names) == 2 and all("T4" in n for n in names), f"Expected two T4s, got {names}"
print("OK:", names, "| torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
# 2. Get the code. Set REPO_URL to your GitHub repository.
REPO_URL = "https://github.com/RTE404/speculative-decoding.git"
REPO_DIR = "/kaggle/working/speculative-decoding"

!rm -rf {REPO_DIR}
!git clone --depth 1 {REPO_URL} {REPO_DIR}
!cd {REPO_DIR} && git log -1 --format="commit %H (%cd)"

In [ ]:
# 3. Install pinned packages. This must run before anything imports transformers:
#    "Save & Run All" cannot restart the kernel. torch is preinstalled and is not reinstalled.
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
# 4. Hugging Face token from Kaggle Secrets, to avoid anonymous download rate limits.
import os
from kaggle_secrets import UserSecretsClient

SECRET_NAME = "kaggle"  # the label of your Kaggle Secret holding the Hugging Face token
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret(SECRET_NAME)
print("HF_TOKEN set")

In [ ]:
# 5. Step 1 check: versions, vocabulary, float16 stability, cache rollback, first estimate of c.
!cd {REPO_DIR} && python -m specdec.check_env

Results are saved in `/kaggle/working/speculative-decoding/results/`, which Kaggle keeps as this notebook's output.